# Imports

In [373]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 10)

# Constants

In [374]:
CD_CONTA = 'CD_CONTA'
DS_CONTA = 'DS_CONTA'

In [375]:
file_name = 'dfp_cia_aberta_BPA_con_2020.csv'
output_name = 'output.csv'

In [376]:
CNPJ_CIA = '97.837.181/0001-47'
ORDEM_EXERC = 'ÚLTIMO'

# User defined functions

In [377]:
"""
def get_drop_mask(merge_df, len_key, min_len):
    can_drop = (merge_df[CD_CONTA] == merge_df.shift(-1)[CD_CONTA])
    return (len_key == min_len) & (can_drop == True)
"""

'\ndef get_drop_mask(merge_df, len_key, min_len):\n    can_drop = (merge_df[CD_CONTA] == merge_df.shift(-1)[CD_CONTA])\n    return (len_key == min_len) & (can_drop == True)\n'

# Read from file

## Load .csv

In [378]:
try:
    cia_aberta_df = pd.read_csv(file_name, encoding='ISO-8859-1', sep=';')
except Exception as e:
    print(f"Error: {e}")

## Select Company

In [379]:
df = cia_aberta_df[cia_aberta_df['CNPJ_CIA'] == CNPJ_CIA].copy()    # seleciono somente as linhas relativas a uma companhia de interesse
df = df[df['ORDEM_EXERC'] == ORDEM_EXERC]                           # seleciono somente o último ou penúltimo exercício
df = df[[CD_CONTA, DS_CONTA,'VL_CONTA']]
df.reset_index(inplace=True, drop=True)

In [380]:
df

,CD_CONTA,DS_CONTA,VL_CONTA
0,1,Ativo Total,11498520.0
1,1.01,Ativo Circulante,4220022.0
2,1.01.01,Caixa e Equivalentes de Caixa,1728413.0
3,1.01.02,Aplicações Financeiras,0.0
4,1.01.02.01,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
...,...,...,...
71,1.02.04.02.07,Goodwill na aquisição da Caetex Florestal,8767.0
72,1.02.04.02.08,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
73,1.02.04.02.09,Goodwill na aquisição da Massima Revestimentos...,6110.0
74,1.02.04.02.10,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


# Wrangling

## Round 1

In [381]:
round = 1

In [382]:
df

,CD_CONTA,DS_CONTA,VL_CONTA
0,1,Ativo Total,11498520.0
1,1.01,Ativo Circulante,4220022.0
2,1.01.01,Caixa e Equivalentes de Caixa,1728413.0
3,1.01.02,Aplicações Financeiras,0.0
4,1.01.02.01,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
...,...,...,...
71,1.02.04.02.07,Goodwill na aquisição da Caetex Florestal,8767.0
72,1.02.04.02.08,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
73,1.02.04.02.09,Goodwill na aquisição da Massima Revestimentos...,6110.0
74,1.02.04.02.10,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### Merge

In [383]:
len_cd = [len(cd) for cd in df[CD_CONTA]]           # len_CD = length of codes in 'CD_CONTA'
min_len = np.array(len_cd).min()            # min_len = minimum length of codes 
ind_min_len = np.where(len_cd==min_len)[0]          # ind_min_len = index of code 'CD_CONTA' with minimum length

In [384]:
df.iloc[ind_min_len]

,CD_CONTA,DS_CONTA,VL_CONTA
0,1,Ativo Total,11498520.0


In [385]:
merge_keys = df[CD_CONTA].apply(lambda string: string[:min_len])
merge_df = pd.merge(merge_keys, df[[CD_CONTA, DS_CONTA]], how='left')

In [386]:
df.insert(round, f'{DS_CONTA}_{round}', merge_df[DS_CONTA])

In [387]:
df

,CD_CONTA,DS_CONTA_1,DS_CONTA,VL_CONTA
0,1,Ativo Total,Ativo Total,11498520.0
1,1.01,Ativo Total,Ativo Circulante,4220022.0
2,1.01.01,Ativo Total,Caixa e Equivalentes de Caixa,1728413.0
3,1.01.02,Ativo Total,Aplicações Financeiras,0.0
4,1.01.02.01,Ativo Total,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
...,...,...,...,...
71,1.02.04.02.07,Ativo Total,Goodwill na aquisição da Caetex Florestal,8767.0
72,1.02.04.02.08,Ativo Total,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
73,1.02.04.02.09,Ativo Total,Goodwill na aquisição da Massima Revestimentos...,6110.0
74,1.02.04.02.10,Ativo Total,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### Drop

For each key which was used to join, we check if the code in the next row in `CD_CONTA` starts with the same key (example: `CD_CONTA` `1.01` starting with `1`; `CD_CONTA` `1.01.01` starting with `1.01`, etc.)

If so, we can safely delete the row.

Otherwise, we keep the row and add `.00` to the code in `CD_CONTA`, so the code can have the correct length in the next round (length 2 in round 2, length 3 in round 3, etc.)

In [388]:
# can_drop = (merge_df[CD_CONTA] == merge_df.shift(-1)[CD_CONTA])
# drop_mask = (len_key == min_len) & (can_drop == True)

for idx in ind_min_len:
    if merge_df.loc[idx, CD_CONTA] == merge_df.loc[idx+1, CD_CONTA]:
        df = df.drop(idx)
    else:
        df.loc[idx, CD_CONTA] = df.loc[idx, CD_CONTA] + '.00'

df.reset_index(inplace=True, drop=True)

In [389]:
# df[drop_mask]

In [390]:
# df = df.drop(df[drop_mask].index)
# df = df.reset_index(drop=True)

## Round 2

In [391]:
round = 2

In [392]:
df

,CD_CONTA,DS_CONTA_1,DS_CONTA,VL_CONTA
0,1.01,Ativo Total,Ativo Circulante,4220022.0
1,1.01.01,Ativo Total,Caixa e Equivalentes de Caixa,1728413.0
2,1.01.02,Ativo Total,Aplicações Financeiras,0.0
3,1.01.02.01,Ativo Total,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,1.01.02.01.01,Ativo Total,Títulos para Negociação,0.0
...,...,...,...,...
70,1.02.04.02.07,Ativo Total,Goodwill na aquisição da Caetex Florestal,8767.0
71,1.02.04.02.08,Ativo Total,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
72,1.02.04.02.09,Ativo Total,Goodwill na aquisição da Massima Revestimentos...,6110.0
73,1.02.04.02.10,Ativo Total,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### Merge

In [393]:
len_cd = [len(cd) for cd in df[CD_CONTA]]           # len_CD = length of codes in 'CD_CONTA'
min_len = np.array(len_cd).min()            # min_len = minimum length of codes 
ind_min_len = np.where(len_cd==min_len)[0]          # ind_min_len = index of code 'CD_CONTA' with minimum length

In [394]:
df.iloc[ind_min_len]

,CD_CONTA,DS_CONTA_1,DS_CONTA,VL_CONTA
0,1.01,Ativo Total,Ativo Circulante,4220022.0
23,1.02,Ativo Total,Ativo Não Circulante,7278498.0


In [395]:
merge_keys = df[CD_CONTA].apply(lambda string: string[:min_len])
merge_df = pd.merge(merge_keys, df[[CD_CONTA, DS_CONTA]], how='left')

In [396]:
merge_df

,CD_CONTA,DS_CONTA
0,1.01,Ativo Circulante
1,1.01,Ativo Circulante
2,1.01,Ativo Circulante
3,1.01,Ativo Circulante
4,1.01,Ativo Circulante
...,...,...
70,1.02,Ativo Não Circulante
71,1.02,Ativo Não Circulante
72,1.02,Ativo Não Circulante
73,1.02,Ativo Não Circulante


In [397]:
df.insert(round, f'{DS_CONTA}_{round}', merge_df[DS_CONTA])

In [398]:
df

,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA,VL_CONTA
0,1.01,Ativo Total,Ativo Circulante,Ativo Circulante,4220022.0
1,1.01.01,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,1728413.0
2,1.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,0.0
3,1.01.02.01,Ativo Total,Ativo Circulante,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,1.01.02.01.01,Ativo Total,Ativo Circulante,Títulos para Negociação,0.0
...,...,...,...,...,...
70,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Caetex Florestal,8767.0
71,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
72,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Massima Revestimentos...,6110.0
73,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### Drop

In [399]:
# can_drop = (merge_df[CD_CONTA] == merge_df.shift(-1)[CD_CONTA])
# drop_mask = (len_key == min_len) & (can_drop == True)

for idx in ind_min_len:
    if merge_df.loc[idx, CD_CONTA] == merge_df.loc[idx+1, CD_CONTA]:
        df = df.drop(idx)
    else:
        df.loc[idx, CD_CONTA] = df.loc[idx, CD_CONTA] + '.00'

df.reset_index(inplace=True, drop=True)

In [400]:
# df[drop_mask]

In [401]:
# df = df.drop(df[drop_mask].index)
# df = df.reset_index(drop=True)

## Round 3

In [402]:
round = 3

In [403]:
df

,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA,VL_CONTA
0,1.01.01,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,1728413.0
1,1.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,0.0
2,1.01.02.01,Ativo Total,Ativo Circulante,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
3,1.01.02.01.01,Ativo Total,Ativo Circulante,Títulos para Negociação,0.0
4,1.01.02.01.02,Ativo Total,Ativo Circulante,Títulos Designados a Valor Justo,0.0
...,...,...,...,...,...
68,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Caetex Florestal,8767.0
69,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
70,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Massima Revestimentos...,6110.0
71,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### Merge

In [404]:
len_cd = [len(cd) for cd in df[CD_CONTA]]           # len_CD = length of codes in 'CD_CONTA'
min_len = np.array(len_cd).min()            # min_len = minimum length of codes 
ind_min_len = np.where(len_cd==min_len)[0]          # ind_min_len = index of code 'CD_CONTA' with minimum length

In [405]:
df.iloc[ind_min_len]

,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA,VL_CONTA
0,1.01.01,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,1728413.0
1,1.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,0.0
7,1.01.03,Ativo Total,Ativo Circulante,Contas a Receber,1318743.0
13,1.01.04,Ativo Total,Ativo Circulante,Estoques,924743.0
14,1.01.05,Ativo Total,Ativo Circulante,Ativos Biológicos,0.0
...,...,...,...,...,...
18,1.01.08,Ativo Total,Ativo Circulante,Outros Ativos Circulantes,71667.0
22,1.02.01,Ativo Total,Ativo Não Circulante,Ativo Realizável a Longo Prazo,2071636.0
46,1.02.02,Ativo Total,Ativo Não Circulante,Investimentos,963437.0
52,1.02.03,Ativo Total,Ativo Não Circulante,Imobilizado,3512641.0


In [406]:
merge_keys = df[CD_CONTA].apply(lambda string: string[:min_len])
merge_df = pd.merge(merge_keys, df[[CD_CONTA, DS_CONTA]], how='left')

In [407]:
merge_df

,CD_CONTA,DS_CONTA
0,1.01.01,Caixa e Equivalentes de Caixa
1,1.01.02,Aplicações Financeiras
2,1.01.02,Aplicações Financeiras
3,1.01.02,Aplicações Financeiras
4,1.01.02,Aplicações Financeiras
...,...,...
68,1.02.04,Intangível
69,1.02.04,Intangível
70,1.02.04,Intangível
71,1.02.04,Intangível


In [408]:
df.insert(round, f'{DS_CONTA}_{round}', merge_df[DS_CONTA])

In [409]:
df

,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA,VL_CONTA
0,1.01.01,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,1728413.0
1,1.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras,0.0
2,1.01.02.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
3,1.01.02.01.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Títulos para Negociação,0.0
4,1.01.02.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Títulos Designados a Valor Justo,0.0
...,...,...,...,...,...,...
68,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Caetex Florestal,8767.0
69,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
70,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Massima Revestimentos...,6110.0
71,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### Drop

In [410]:
# can_drop = (merge_df[CD_CONTA] == merge_df.shift(-1)[CD_CONTA])
# drop_mask = (len_key == min_len) & (can_drop == True)

for idx in ind_min_len:
    if merge_df.loc[idx, CD_CONTA] == merge_df.loc[idx+1, CD_CONTA]:
        df = df.drop(idx)
    else:
        df.loc[idx, CD_CONTA] = df.loc[idx, CD_CONTA] + '.00'

df.reset_index(inplace=True, drop=True)

In [411]:
# df[drop_mask]

In [413]:
# df = df.drop(df[drop_mask].index)
# df = df.reset_index(drop=True)

In [414]:
df

,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA,VL_CONTA
0,1.01.02.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
1,1.01.02.01.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Títulos para Negociação,0.0
2,1.01.02.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Títulos Designados a Valor Justo,0.0
3,1.01.02.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,1.01.02.03,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas ao Custo Amor...,0.0
...,...,...,...,...,...,...
59,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Caetex Florestal,8767.0
60,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
61,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Massima Revestimentos...,6110.0
62,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


## Round 4

In [415]:
round = 4

In [416]:
df

,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA,VL_CONTA
0,1.01.02.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
1,1.01.02.01.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Títulos para Negociação,0.0
2,1.01.02.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Títulos Designados a Valor Justo,0.0
3,1.01.02.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,1.01.02.03,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas ao Custo Amor...,0.0
...,...,...,...,...,...,...
59,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Caetex Florestal,8767.0
60,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
61,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Massima Revestimentos...,6110.0
62,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### Merge

In [417]:
len_cd = [len(cd) for cd in df[CD_CONTA]]           # len_CD = length of codes in 'CD_CONTA'
min_len = np.array(len_cd).min()            # min_len = minimum length of codes 
ind_min_len = np.where(len_cd==min_len)[0]          # ind_min_len = index of code 'CD_CONTA' with minimum length

In [418]:
df.iloc[ind_min_len]

,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA,VL_CONTA
0,1.01.02.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
3,1.01.02.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,1.01.02.03,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas ao Custo Amor...,0.0
5,1.01.03.01,Ativo Total,Ativo Circulante,Contas a Receber,Clientes,1239315.0
8,1.01.03.02,Ativo Total,Ativo Circulante,Contas a Receber,Outras Contas a Receber,79428.0
...,...,...,...,...,...,...
45,1.02.03.01,Ativo Total,Ativo Não Circulante,Imobilizado,Imobilizado em Operação,3377237.0
46,1.02.03.02,Ativo Total,Ativo Não Circulante,Imobilizado,Direito de Uso em Arrendamento,0.0
47,1.02.03.03,Ativo Total,Ativo Não Circulante,Imobilizado,Imobilizado em Andamento,135404.0
48,1.02.04.01,Ativo Total,Ativo Não Circulante,Intangível,Intangíveis,406628.0
